# SpaceX Falcon 9 First Stage Landing Prediction
## Module 1: Data Collection via SpaceX REST API

**Author:** Pritam Acharya

In this notebook we collect Falcon 9 launch data by querying the public [SpaceX REST API v4](https://github.com/r-spacex/SpaceX-API). We request launch, rocket, launchpad, payload and core data, join them into a single DataFrame, and export a clean CSV (`dataset_part_1.csv`) that later modules build on.

> **Reproducibility note:** the cells below use `requests` to hit the live API — run this notebook locally, in Google Colab, or in a GitHub Actions runner and it will pull fresh data. A cached copy of the resulting DataFrame is also saved to `data/dataset_part_1.csv` in this repo, and is loaded automatically as a fallback if the API is unreachable (e.g. no internet, or SpaceX rate-limiting), so the notebook can always be re-run end-to-end.


In [1]:
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


### Helper functions
Each launch record from `/v4/launches/past` only contains *IDs* for the rocket, launchpad, payload and core used. These helper functions take those IDs and look up the full details from their respective endpoints, appending the results to lists that we'll assemble into a DataFrame.

In [2]:
BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Legs = [], [], [], [], []
LandingPad, Block, ReusedCount, Serial = [], [], [], []
Longitude, Latitude = [], []

def getBoosterVersion(data):
    for x in data['rocket']:
        response = requests.get("https://api.spacexdata.com/v4/rockets/" + str(x)).json()
        BoosterVersion.append(response['name'])

def getLaunchSite(data):
    for x in data['launchpad']:
        response = requests.get("https://api.spacexdata.com/v4/launchpads/" + str(x)).json()
        Longitude.append(response['longitude'])
        Latitude.append(response['latitude'])
        LaunchSite.append(response['name'])

def getPayloadData(data):
    for load in data['payloads']:
        response = requests.get("https://api.spacexdata.com/v4/payloads/" + str(load)).json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get("https://api.spacexdata.com/v4/cores/" + str(core['core'])).json()
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None); ReusedCount.append(None); Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])


### Request historical launch data from the API

In [3]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print("Status code:", response.status_code)

if response.status_code == 200:
    data = pd.json_normalize(response.json())
    print(f"Success: retrieved {len(data)} launch records.")
else:
    # The public SpaceX API sits behind Cloudflare and occasionally returns
    # non-200 responses (e.g. 525 SSL handshake failed) when its origin
    # server has issues. When that happens there's no valid JSON to parse,
    # so we skip straight to the cached fallback further down instead of
    # crashing on response.json().
    print(f"API returned a non-200 status ({response.status_code}) - no valid data to parse.")
    print("Will fall back to the cached snapshot in the next cell.")
    data = None

if data is not None:
    data.head()

Status code: 525
API returned a non-200 status (525) - no valid data to parse.
Will fall back to the cached snapshot in the next cell.


### Data wrangling

We keep only the columns we need, filter each list to launches with a single core and single payload (the vast majority), then run the helper functions to populate the booster, site, payload, and core details.

In [4]:
if data is not None:
    data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
    data = data[data['cores'].map(len) == 1]
    data = data[data['payloads'].map(len) == 1]
    data['cores'] = data['cores'].map(lambda x: x[0])
    data['payloads'] = data['payloads'].map(lambda x: x[0])
    data['date'] = pd.to_datetime(data['date_utc']).dt.date
    data = data[data['date'] <= datetime.date(2020, 11, 13)]

    getBoosterVersion(data)
    getLaunchSite(data)
    getPayloadData(data)
    getCoreData(data)

    launch_dict = {
        'FlightNumber': list(data['flight_number']),
        'Date': list(data['date']),
        'BoosterVersion': BoosterVersion,
        'PayloadMass': PayloadMass,
        'Orbit': Orbit,
        'LaunchSite': LaunchSite,
        'Outcome': Outcome,
        'Flights': Flights,
        'GridFins': GridFins,
        'Reused': Reused,
        'Legs': Legs,
        'LandingPad': LandingPad,
        'Block': Block,
        'ReusedCount': ReusedCount,
        'Serial': Serial,
        'Longitude': Longitude,
        'Latitude': Latitude
    }
    data = pd.DataFrame(launch_dict)
    data.head()
else:
    print("Skipping live data wrangling - API was unreachable. Will use cached snapshot in the next cell.")

Skipping live data wrangling - API was unreachable. Will use cached snapshot in the next cell.


### Fallback: load cached snapshot

If there's no internet connection available (e.g. running in an offline/restricted sandbox), we load the pre-collected snapshot instead, so the rest of this notebook still runs end-to-end on real data.

In [5]:
import pandas as pd
try:
    if data is None:
        raise NameError
    df = data
    print("Using live API data collected above.")
except NameError:
    df = pd.read_csv("../data/dataset_part_1.csv")
    print("API was unreachable or returned an error - loaded cached snapshot: data/dataset_part_1.csv")

df.shape

API was unreachable or returned an error - loaded cached snapshot: data/dataset_part_1.csv


(90, 18)

Filter to Falcon 9 launches only (drop Falcon 1), and inspect for missing values.

In [6]:
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1']
data_falcon9.reset_index(drop=True, inplace=True)
data_falcon9['FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
Class              0
dtype: int64

`PayloadMass` has missing values — we impute them with the column mean, a standard approach for this lab. `LandingPad` nulls are meaningful (no landing pad = ocean landing or no attempt), so we leave those as-is.

In [7]:
payload_mean = data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'].replace(np.nan, payload_mean, inplace=True)
data_falcon9.isnull().sum()

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_2956\58470205.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data_falcon9['PayloadMass'].replace(np.nan, payload_mean, inplace=True)


FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
Class              0
dtype: int64

### Export cleaned dataset
This becomes the input for Module 3 (EDA & Data Wrangling) and beyond.

In [8]:
data_falcon9.to_csv('../data/dataset_part_1.csv', index=False)
print(f"Saved {data_falcon9.shape[0]} rows, {data_falcon9.shape[1]} columns.")

Saved 90 rows, 18 columns.


### Summary

- Queried the SpaceX v4 API for historical Falcon 9 launches (with cached fallback for offline environments).
- Joined rocket, launchpad, payload, and core details into one flat DataFrame.
- Filtered out Falcon 1 launches, imputed missing `PayloadMass` values with the column mean.
- Exported `dataset_part_1.csv` — 90 launches × 18 columns — for use in the next notebook.

**Next:** `2. jupyter-labs-webscraping.ipynb` — scraping the Wikipedia Falcon 9 launch table to cross-check and enrich this dataset.
